[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Laboratorio 5: Modelos en PyTorch
## Parte 3: Clasificación Multiclase (Dataset Multimodal - Audio)
Para esta última parte utilizaremos el dataset **FMA** (Free Music Archive), el cual cumple con la condición de ser un dataset multimodal (audio, aunque extraeremos sus características acústicas/metadatos), tener más de 100 propiedades ($n > 100$) y más de 5000 ejemplos ($m \ge 5000$).

El objetivo de esta red es clasificar a qué **género musical** pertenece una pista en base a sus propiedades acústicas (MFCC, contrastes espectrales, etc.).

In [1]:
# IMPORTANTE: Si estás ejecutando este cuadernillo en Google Colab, 
# descomenta y ejecuta estas líneas para montar tu Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


### 3.1 Carga del Dataset y Preprocesamiento
El dataset FMA puede tener diversas estructuras (archivos separados para metadata y características, múltiples filas de encabezado, etc.). Aquí asumiremos que hemos cargado un `.csv` plano o procesaremos el archivo de características.

In [2]:
# FMA Limpio y Balanceado
# Este archivo contiene todas las características musicales numéricas 
# y en la última columna la etiqueta oficial del Género Musical ya balanceada.
archivo_fma = 'fma_limpio_balanceado.csv'

# Al ser un CSV limpio, lo cargamos de manera normal
df_fma = pd.read_csv(archivo_fma, low_memory=False)

# Separamos X (propiedades) y Y (Género)
X_pandas = df_fma.iloc[:, :-1].values 
Y_pandas_raw = df_fma.iloc[:, -1].values 

# Convertimos las palabras del género ('Rock', 'Pop', etc.) a enteros (0, 1, 2, C-1)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
Y_pandas_encoded = le.fit_transform(Y_pandas_raw)

m = len(Y_pandas_encoded)
n = X_pandas.shape[1]
num_clases = len(np.unique(Y_pandas_encoded))

print(f"Total de ejemplos (m): {m} ")
print(f"Total de propiedades (n): {n} ")
print(f"Clases identificadas: {num_clases} -> Corresponde a {le.classes_}")

FileNotFoundError: [Errno 2] No such file or directory: 'fma_limpio_balanceado.csv'

### 3.2 División 80/20, Normalización y DataLoaders
Al tener muchas más propiedades (cientos) y datos en diferenes escalas (algunos en Hertz, otros con valores negativos como MFCCs), la estandarización ($z-score$) es **absolutamente crucial** para evitar el famoso *exploding gradient* o valores nulos (`NaN`).

In [ ]:
limite_fma = int(0.8 * m)

# Mezclamos aleatoriamente los datos antes del split para evitar sesgos si vienen ordenados
indices = np.random.permutation(m)
X_pandas = X_pandas[indices]
Y_pandas_encoded = Y_pandas_encoded[indices]

X_train_raw = X_pandas[:limite_fma]
X_test_raw = X_pandas[limite_fma:]
Y_train = Y_pandas_encoded[:limite_fma]
Y_test = Y_pandas_encoded[limite_fma:]

# Normalización Z-Score
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0) + 1e-8  # epsilon para evitar divisiones entre 0

X_train = (X_train_raw - X_mean) / X_std
X_test = (X_test_raw - X_mean) / X_std

# ¡Para multiclase (CrossEntropyLoss en PyTorch) Y debe ser LongTensor sin .float() ni dimensión de columna!
X_train_tensor = torch.from_numpy(X_train).float()
Y_train_tensor = torch.from_numpy(Y_train).long() # <-- long() y array de una dimensión (D,)

X_test_tensor = torch.from_numpy(X_test).float()
Y_test_tensor = torch.from_numpy(Y_test).long()

class DatasetMulticlase(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        self.X = X.to(device)
        self.Y = Y.to(device)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

dataset_train = DatasetMulticlase(X_train_tensor, Y_train_tensor)
dataset_test = DatasetMulticlase(X_test_tensor, Y_test_tensor)

batch_size = 500 # Podemos usar baches más grandes si hay mucha información
dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size, shuffle=False)
print("DataLoaders listos.")

### 3.3 Arquitectura de la Red Multiclase
Para la tarea muticlase, configuraremos una red neuronal profunda.

**Consideración Clave:**
En PyTorch, la función `nn.CrossEntropyLoss()` internamente aplica `LogSoftmax` y `NLLLoss`. Es decir, la salida de nuestro modelo debe ser de la forma $(Batch, Numero\_de\_Clases)$ y **no debemos aplicar manualmente la función `Softmax` al final de la capa**. Emitiremos los valores "crudos" o *logits*.

In [ ]:
D_in = n              # Entrada = Cantidad de características (> 100)
H1 = 250              # Capa Oculta 1 (Gran cantidad debido a la dimensionalidad mayor)
H2 = 100              # Capa Oculta 2
D_out = num_clases    # Salida = Número de Clases (géneros) detectadas

model_multi = nn.Sequential(
    nn.Linear(D_in, H1),
    nn.ReLU(),
    nn.Linear(H1, H2),
    nn.ReLU(),
    nn.Linear(H2, D_out)    # ¡SIN FUNCIÓN DE ACTIVACIÓN AL FINAL! CrossEntropyLoss se encarga.
).to(device)

criterion_multi = nn.CrossEntropyLoss()
# Adam suele funcionar excelente para dimensionalidades altas como audio/texto
optimizer_multi = torch.optim.Adam(model_multi.parameters(), lr=0.001)

epochs = 150
log_each = 30
l_multi = []
model_multi.train()

print("Iniciando Entrenamiento (Multiclase FMA)...")
for e in range(1, epochs + 1):
    _l = []
    for x_b, y_b in dataloader_train:
        y_pred = model_multi(x_b)           # Logits de salida (m, D_out)
        loss = criterion_multi(y_pred, y_b) # Ocupamos CrossEntropyLoss
        
        _l.append(loss.item())
        optimizer_multi.zero_grad()
        loss.backward()
        optimizer_multi.step()

    l_multi.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} - Costo (CrossEntropyLoss): {np.mean(_l):.5f}")

PATH_MULTI = './checkpoint_clasificacion_multiclase.pt'
torch.save(model_multi.state_dict(), PATH_MULTI)
print(f"Modelo guardado en: {PATH_MULTI}")

plt.plot(range(1, epochs + 1), l_multi, color='purple')
plt.title('Gráfico de Costo - Clasificación Multiclase (FMA)')
plt.xlabel('Épocas')
plt.ylabel('Costo (CrossEntropy)')
plt.grid(True)
plt.show()

### 3.4 Predicción y Precisión General del Modelo
Para predecir, tomamos el valor máximo del vector de logits resultante usando `torch.max()`. Este valor máximo indica la clase que la red neuronal cree que es más probable.

In [ ]:
model_multi.eval() 

correctos = 0
total = 0

with torch.no_grad():
    for x_b, y_b in dataloader_test:
        logits = model_multi(x_b)
        # torch.max devuelve el valor máximo y su índice (la clase ganadora). Nos importa el índice -> [1]
        _, prediccion_clase = torch.max(logits, 1) 
        
        total += y_b.size(0)
        correctos += (prediccion_clase == y_b).sum().item()

accuracy = (correctos / total) * 100
print(f"La Precisión Multiclase (Accuracy general) alcanzada es: {accuracy:.2f}%\n")

# Extraer 15 ejemplos para probar
con_ejemplos = 15
X_prueba = X_test_tensor[:con_ejemplos].to(device)
Y_prueba = Y_test_tensor[:con_ejemplos].to(device)

with torch.no_grad():
    logits_15 = model_multi(X_prueba)
    _, predicciones_15 = torch.max(logits_15, 1)

print(f"--- Evaluando {con_ejemplos} Muestras Especiales ---")
for i in range(con_ejemplos):
    real = le.inverse_transform([Y_prueba[i].cpu().item()])[0]         # Regresamos el label original desde el número
    predicho = le.inverse_transform([predicciones_15[i].cpu().item()])[0]
    flag = "✅ Correcto" if real == predicho else "❌ Fallo"
    print(f"Fila {i+1} | Género Real: {real: <15} | Predicción: {predicho: <15} {flag}")